# Phase 2 — Strategy Engine + Real-Log Validation

Faithful bar-by-bar Python reimplementation of the FYP IFVG+CISD NQ strategy,
validated against two real TradingView trade logs.

**Honest headline (read this first):** run with the Pine script's *default*
parameters, the engine reproduces **76-80% of the real logs' entry-days and
directions** (good recall), but it **over-generates roughly 4x as many
trades** (precision only 20-25%) and **underperforms both real logs on
profit factor and win rate**. It is directionally consistent on both a
losing regime and a winning regime (generated PF < 1 when real PF < 1;
generated PF > 1 when real PF > 1), which is evidence the core strategy
mechanic is faithfully ported — but the real "optimised" track record's
edge over this raw signal appears to come substantially from parameter
tuning / selectivity that the default parameters do not capture. See
`WRITEUP_STRATEGY.md` (repo root) for the full writeup, including caveats
about data comparability that must be read before trusting any absolute
price or dollar figure below.

## 1. Setup

In [ ]:
import os
import sys
import time
from datetime import date
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

# Make relative paths (data/raw/..., charts/...) resolve correctly whether
# Jupyter was launched from the repo root or from notebooks/.
if Path.cwd().name == "notebooks":
    os.chdir(Path.cwd().parent)
assert (Path.cwd() / "run_backtest.py").exists(), f"expected repo root, got {Path.cwd()}"
print(f"Working directory: {Path.cwd()}")

# Explicitly (not just implicitly via IPython's cwd convention) put the repo
# root on sys.path so `import nqdata` / `import backtest` / etc. resolve
# regardless of kernel/launcher.
if str(Path.cwd()) not in sys.path:
    sys.path.insert(0, str(Path.cwd()))

from nqdata.load import load_nq
from backtest.engine import backtest
from validate_trades import parse_tv_log, compare
from metrics import profit_factor, win_rate, total_pnl, max_drawdown

## 2. Load data

The Phase-1 raw dataset: ~3 years of 1-minute NQ OHLCV bars, back-adjusted
continuous contract, `US/Eastern`-indexed.

In [ ]:
t0 = time.time()
df = load_nq()
print(f"{len(df):,} rows, {df.index[0]} .. {df.index[-1]}  ({time.time() - t0:.1f}s)")
df.head()

## 3. Run the backtest

Bar-by-bar, pure-Python loop over the full ~1.05M-row series (session gate,
IFVG, CISD, EMA filter, double-confirmation entry, 8-bar swing stop, 1.5R
target, 1 trade/day, stop-first exit resolution). **This takes about 20
seconds** — there is no vectorized shortcut here, by design: every bar's
decision must see only bars up to and including itself (no lookahead).

In [ ]:
t1 = time.time()
trades = backtest(df)
same_bar_span_count = backtest.same_bar_span_count
print(f"{len(trades)} generated trades, {same_bar_span_count} same-bar-span exits "
      f"({time.time() - t1:.1f}s)")

## 4. Generated aggregate (all trades, unfiltered)

In [ ]:
pnls = [tr.pnl_usd for tr in trades]
generated_aggregate = {
    "total_trades": len(trades),
    "profit_factor": profit_factor(pnls),
    "win_rate": win_rate(pnls),
    "total_pnl": total_pnl(pnls),
    "max_drawdown": max_drawdown(pnls),
    "long": sum(1 for tr in trades if tr.direction == "Long"),
    "short": sum(1 for tr in trades if tr.direction == "Short"),
}
pd.Series(generated_aggregate)

## 5. Validate against the two real trade logs

Match key: **(entry NY session-date, direction)** only — never absolute
price (see the data-comparability caveat in `WRITEUP_STRATEGY.md`). Both the
generated set and each real log are window-clipped to that log's own
coverage window before comparing, so the two disjoint logs (2023-24 vs
2025-26) never see each other's generated trades, and out-of-coverage real
trades are excluded and counted rather than scored as misses.

In [ ]:
REAL_LOG_PATHS = {
    "2023_2024": "C:/Users/Alex/Projects/Trading-Strategy-Monte-Carlo-Simulation/data/NQ1_2023_2024.csv",
    "optimised": "C:/Users/Alex/Projects/Trading-Strategy-Monte-Carlo-Simulation/data/NQ1_optimised.csv",
}
# Phase-1 raw-data coverage bounds (see docs/superpowers/plans/2026-07-13-phase2-strategy-engine.md).
DATA_WINDOW_START = date(2022, 12, 26)
DATA_WINDOW_END = date(2025, 12, 11)

real_logs = {name: parse_tv_log(path) for name, path in REAL_LOG_PATHS.items()}

windows = {}
reports = {}
for name, real_df in real_logs.items():
    win_start = max(real_df["entry_date"].min(), DATA_WINDOW_START)
    win_end = min(real_df["entry_date"].max(), DATA_WINDOW_END)
    windows[name] = (win_start, win_end)
    reports[name] = compare(trades, real_df, win_start, win_end)

list(reports.keys())

### Coverage table

`real_in_window` is the true backtestable baseline for that log (note: the
`optimised` log's full real total is 72 trades / +$28,400 — 13 of those
trades, +$10,285, fall after the data's 2025-12-11 edge and are correctly
excluded here rather than silently compared against).

In [ ]:
coverage_rows = []
for name, r in reports.items():
    coverage_rows.append({
        "log": name,
        "window": f"{r['win_start']} .. {r['win_end']}",
        "real_in_window": r["n_real_in_window"],
        "real_excluded": r["n_real_excluded"],
        "generated_in_window": r["n_generated_in_window"],
        "matched": r["n_matched"],
        "missed": r["n_missed"],
        "extra": r["n_extra"],
        "precision": round(r["precision"], 3),
        "recall": round(r["recall"], 3),
    })
coverage_df = pd.DataFrame(coverage_rows).set_index("log")
coverage_df

### Generated vs real aggregate (same window, per log)

Both sides computed over the *same* window-clipped trades, so this is an
apples-to-apples PF / win-rate / PnL comparison of "what the raw signal
would have done" vs "what the real (tuned/selective) track record did."

In [ ]:
agg_rows = []
for name, r in reports.items():
    for side in ("generated", "real"):
        a = r["aggregate"][side]
        pf = a["profit_factor"]
        agg_rows.append({
            "log": name,
            "side": side,
            "profit_factor": round(pf, 3) if pf != float("inf") else pf,
            "win_rate": round(a["win_rate"], 3),
            "total_pnl": a["total_pnl"],
            "long": a["direction_mix"]["Long"],
            "short": a["direction_mix"]["Short"],
        })
agg_df = pd.DataFrame(agg_rows).set_index(["log", "side"])
agg_df

### Out-of-coverage generated trades (reported, not hidden)

Generated trades whose entry falls outside *both* logs' windows are never
scored as false positives against either log (they're clipped out of both
`compare()` calls) — but they should not silently vanish from the total
either, so they're counted here explicitly.

In [ ]:
def _entry_date(tr):
    ts = pd.Timestamp(tr.entry_time)
    if ts.tzinfo is not None:
        ts = ts.tz_convert("America/New_York")
    return ts.date()

(first_start, first_end), (second_start, second_end) = sorted(windows.values())

before_first, inter_gap, after_last = [], [], []
for tr in trades:
    d = _entry_date(tr)
    if (first_start <= d <= first_end) or (second_start <= d <= second_end):
        continue
    if d < first_start:
        before_first.append(d)
    elif d > second_end:
        after_last.append(d)
    else:
        inter_gap.append(d)

pd.Series({
    "total_out_of_window": len(before_first) + len(inter_gap) + len(after_last),
    "before_first_window": len(before_first),
    "inter_log_gap": len(inter_gap),
    "after_last_window": len(after_last),
})

## 6. Charts

Generated by `run_backtest.py` (re-run `python run_backtest.py` to
regenerate); loaded here from `charts/` to narrate alongside the tables
above.

### Equity curve — cumulative P&L across all 605 generated trades

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.imshow(mpimg.imread("charts/equity_curve.png"))
ax.axis("off")
plt.show()

### Coverage bars — real (in-window) / matched / missed / extra, per log

The `extra` bar towers over `real_in_window` in both logs — the visual
version of the ~20-25% precision figures above: the raw default-parameter
signal fires far more often than the real, selective track record traded.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
ax.imshow(mpimg.imread("charts/coverage.png"))
ax.axis("off")
plt.show()

### Generated vs real — profit factor & win rate, per log

Generated PF/WR sits below real PF/WR on both logs, but on the correct side
of 1.0 / above chance on both — directionally honest, not fitted.

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))
ax.imshow(mpimg.imread("charts/generated_vs_real_pf_wr.png"))
ax.axis("off")
plt.show()

## 7. Honest conclusion

- The double-confirmation IFVG+CISD entry logic, run with Pine's default
  parameters and no fitting to either log, recovers the **majority of the
  real strategy's trade-days and directions** (76-80% recall) — meaningful
  evidence the ported strategy logic is substantively correct, given the
  ports were never validated against a live TradingView chart (no premium
  subscription available — see caveat in `WRITEUP_STRATEGY.md`).
- It also **fires about 4x more often than the real track record**
  (20-25% precision) and **underperforms on PF and win rate on both the
  losing 2023-24 period and the winning 2025-26 period** — the real
  "optimised" edge is not simply "run the base strategy"; a meaningful part
  of it comes from tuning/selectivity this rebuild deliberately does not
  add (Global Constraints: `fill_mode` and every threshold are fixed
  a-priori, never chosen to agree with the logs).
- Absolute entry prices and dollar PnL are **not** directly comparable
  between generated and real trades — the data is a back-adjusted
  continuous NQ series, the logs are unadjusted front-month prints. The
  headline metrics here (date+direction coverage, win/loss outcome) are
  offset-invariant; see `WRITEUP_STRATEGY.md` for the full caveat.
- This gap is the direct motivation for **Phase 4**: a parameter sweep /
  regime filter to search for the region of parameter space the real
  "optimised" track record likely lived in, rather than assuming the
  Pine defaults were ever the live-traded parameters.

Full writeup, caveats, and the corrected CISD off-by-one bug: see
`WRITEUP_STRATEGY.md` in the repo root.